# Olist: GMV by product category

## Goal
Which product categories account for most of the item-price GMV? Check the products and category-translation tables first, then compute GMV and share by category. The SQL version is in `sql/06_category_gmv_rank.sql`.

## Setup
Four tables: orders, order items, products and the category translation.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

# Project folder = the nearest parent that contains data/raw
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
DATA_DIR = ROOT / "data" / "raw"

orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
products = pd.read_csv(DATA_DIR / "olist_products_dataset.csv")
translation = pd.read_csv(DATA_DIR / "product_category_name_translation.csv")

print("orders:", orders.shape)
print("items:", items.shape)
print("products:", products.shape)
print("translation:", translation.shape)

orders: (99441, 8)
items: (112650, 7)
products: (32951, 9)
translation: (71, 2)


## 1. Checking products and the translation table
- What does one row of `products` represent, and is `product_id` unique?
- How many products have no `product_category_name`?
- Do all categories in `products` exist in the translation table?
- Does every `product_id` in `items` exist in `products`?

In [2]:
display(products.head(3))
display(translation.head(3))

# 1. Duplicate product_id
print("Duplicate product_id:", products["product_id"].duplicated().sum())

# 2. Products without a category
print("Missing category:", products["product_category_name"].isna().sum())

# 3. Categories in products that are absent from the translation table
cats = pd.Series(products["product_category_name"].dropna().unique())
not_translated = cats[~cats.isin(translation["product_category_name"])]
print("Distinct non-missing categories:", len(cats))
print("Categories without a translation:", list(not_translated))

# 4. Item rows whose product_id is not in products
print("Item rows without a matching product:", (~items["product_id"].isin(products["product_id"])).sum())

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


Duplicate product_id: 0
Missing category: 610
Distinct non-missing categories: 73
Categories without a translation: ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Item rows without a matching product: 0


## 2. GMV by category
Same definitions as monthly GMV: GMV = sum of `price`, valid orders only (canceled and unavailable excluded).

- Items of valid orders are joined to products to get each item's category; the item row count must not change.
- `price` is summed by category and sorted descending.
- Products without a category are kept as their own row, `(missing)`, not dropped.

Check: the categories must add up to 13,494,400.74.

In [3]:
# 1. order_id of valid orders
valid_ids = orders.loc[~orders["order_status"].isin(["canceled", "unavailable"]), "order_id"]

# 2. Items of valid orders only
valid_items = items[items["order_id"].isin(valid_ids)]
print("Item rows of valid orders:", len(valid_items))

# 3. Left join to products to get the category; every item row is kept
merged = valid_items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
print("Rows after join:", len(merged))          # must equal the line above

# 4. Name the missing category so that groupby keeps it
merged["category"] = merged["product_category_name"].fillna("(missing)")

# 5. Sum price by category, descending
category_gmv = merged.groupby("category")["price"].sum().sort_values(ascending=False)

# 6. Table with GMV and share
result = category_gmv.reset_index(name="gmv")
result["pct"] = (result["gmv"] / result["gmv"].sum() * 100).round(1)

print("Categories (including missing):", len(result))
print("GMV total:", round(result["gmv"].sum(), 2))     # expect 13494400.74
display(result.head(10))
print("Share of the top 10:", round(result["gmv"].head(10).sum() / result["gmv"].sum() * 100, 1))
display(result[result["category"] == "(missing)"])

Item rows of valid orders: 112101
Rows after join: 112101
Categories (including missing): 74
GMV total: 13494400.74


,category,gmv,pct
0,beleza_saude,1255695.13,9.3
1,relogios_presentes,1198185.21,8.9
2,cama_mesa_banho,1035964.06,7.7
3,esporte_lazer,979740.92,7.3
4,informatica_acessorios,904322.02,6.7
5,moveis_decoracao,727465.05,5.4
6,utilidades_domesticas,626825.80,4.6
7,cool_stuff,620770.49,4.6
8,automotivo,586585.73,4.3
9,ferramentas_jardim,481009.94,3.6


Share of the top 10: 62.4


,category,gmv,pct
20,(missing),178572.55,1.3


## Checks
- The item row count is the same before and after the join (112,101).
- The categories add up to 13,494,400.74.
- The missing category has its own row.
- Ties: no two categories have the same GMV, so the ranking is unambiguous; the SQL version uses `RANK()`, which would give tied categories the same rank.

## Findings
- There are 73 categories plus one group of products without a category.
- The top three are beleza_saude (health_beauty) with 1,255,695.13 (9.3%), relogios_presentes (watches_gifts) with 1,198,185.21 (8.9%) and cama_mesa_banho (bed_bath_table) with 1,035,964.06 (7.7%).
- The top 10 categories account for 62.4% of GMV.
- Products without a category add up to 178,572.55, or 1.3%, and rank 21st.
- Ranking by GMV is not ranking by units sold or by profit: categories with high unit prices rank high on value without necessarily selling many items.